# QLoRA SFT — Qwen3-4B-Thinking on OpenR1-Math-220k

Notebook version of `train_lora.py`. Same recipe, same defaults; run cell-by-cell so you can inspect the dataset and intermediate state before kicking off the long training step.

**Recipe** (finalized 2026-05-28, see `todo.md` §A3.5 for sources):
- QLoRA 4-bit nf4 + LoRA r=16/α=16 on `q/k/v/o/gate/up/down`
- LR 2e-4, `cosine_with_min_lr` (min 0.1), warmup 0.03, 2 epochs
- batch=1, grad_accum=16 (effective 16), max_seq=8192, packing=false
- max_grad_norm=0.2, weight_decay=0.0, Unsloth 8-bit AdamW
- `train_on_responses_only` (loss masked to assistant tokens)
- Dataset: OpenR1-Math-220k `default` filtered to math-verified + reasoning-complete, random 30k subsample (seed=42)

**Hardware**: 24 GB GPU sufficient (A5000 / RTX 4090 / Blackwell MIG / A100). bf16-capable card required (Ampere+).

**Order of operations**: run cells top-to-bottom. The cells are split so you can re-run the dataset / trainer setup without re-loading the model (the model load is the slowest non-training step).


## 1. Environment variables — set BEFORE the first `import torch`

If you re-run this notebook after already importing torch in this kernel, **restart the kernel** for these to take effect.


In [ ]:
import os

# expandable_segments lets the CUDA allocator grow segments instead of pre-reserving
# fixed chunks — important for the 32k-token tail of OpenR1 samples.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("UNSLOTH_RETURN_LOGITS", "0")
print("Env vars set. If torch was already imported in this kernel, restart it now.")


## 2. Imports + GPU sanity check

Expected on DSMLP A5000: `bf16: True`, device `NVIDIA RTX A5000`. If `bf16: False` you've landed on a Turing card (T4, 2080Ti) — re-launch the pod, Unsloth kernels won't work.


In [ ]:
import torch
from pathlib import Path

from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

print(f"torch {torch.__version__}, CUDA {torch.version.cuda}")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")
print(f"device: {torch.cuda.get_device_name(0)}")


## 3. Configuration

All knobs at the top so you can audit the recipe by reading 30 lines. The three flags at the very top control run mode:

- `SMOKE_TEST = True` — 200 examples, `max_steps=10` (~10 min sanity check)
- `RESUME = True` — resume from the latest checkpoint in `OUTPUT_DIR`
- `PUSH_TO = '<user>/<repo>'` — push final adapter to HF Hub after training; `None` to skip


In [ ]:
# ===== Run mode =====
SMOKE_TEST = False     # True = 200 examples, max_steps=10 (~10 min sanity check)
RESUME     = False     # True = resume from latest checkpoint in OUTPUT_DIR
PUSH_TO    = None      # e.g. "yourname/qwen3-4b-math-lora"  to push at end; None to skip

# ===== Model / data =====
MODEL_NAME      = "Qwen/Qwen3-4B-Thinking-2507"
DATASET_NAME    = "open-r1/OpenR1-Math-220k"
DATASET_CONFIG  = "default"   # 94k — best SFT performance per dataset card
SUBSAMPLE_SIZE  = 30_000
SEED            = 42
MAX_SEQ_LEN     = 8192
OUTPUT_DIR      = "./checkpoints/sft_qwen3_4b"

# ===== LoRA =====
LORA_R          = 16
LORA_ALPHA      = 16
LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

# ===== Training (see todo.md §A3.5 for source reconciliation) =====
LR               = 2e-4
WARMUP_RATIO     = 0.03
NUM_EPOCHS       = 2
PER_DEVICE_BS    = 1
GRAD_ACCUM       = 16          # effective batch 16
MAX_GRAD_NORM    = 0.2
WEIGHT_DECAY     = 0.0
SAVE_STEPS       = 200
SAVE_TOTAL_LIMIT = 3

# Instruction matches R1's generation prompt — keeps train/test distribution aligned.
USER_INSTRUCTION = "\n\nPlease reason step by step, and put your final answer within \\boxed{}."


## 4. Dataset-prep helpers

`pick_valid_generation` picks the first of the 1-6 R1 traces per problem that's both math-verified AND complete. `build_formatter` is a closure capturing the tokenizer so we can use it with `Dataset.map`.


In [ ]:
def pick_valid_generation(row):
    """Index of first generation that is BOTH math-verified AND reasoning-complete, else None."""
    correctness  = row.get("correctness_math_verify") or []
    completeness = row.get("is_reasoning_complete")  or []
    gens         = row.get("generations")            or []
    for i in range(len(gens)):
        ok_math = i < len(correctness)  and bool(correctness[i])
        ok_done = i < len(completeness) and bool(completeness[i])
        if ok_math and ok_done:
            return i
    return None


def build_formatter(tokenizer):
    """Closure: row -> {'text': <chat-templated str>} or {'text': None} (drop)."""
    def format_row(row):
        idx = pick_valid_generation(row)
        if idx is None:
            return {"text": None}
        problem = row.get("problem") or row.get("question")
        if not problem:
            return {"text": None}
        generation = row["generations"][idx]
        if not generation or not generation.strip():
            return {"text": None}
        messages = [
            {"role": "user",      "content": problem + USER_INSTRUCTION},
            {"role": "assistant", "content": generation},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False,
        )
        return {"text": text}
    return format_row


## 5. Model + tokenizer load

~2 min on a fresh pod (downloads ~8 GB model). Cached afterward.


In [ ]:
print(f"Loading {MODEL_NAME} in 4-bit ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,             # auto-detect (bf16 on Ampere/Hopper/Blackwell)
    load_in_4bit=True,
)


## 6. Attach LoRA adapters

Wraps the frozen 4-bit base with trainable rank-16 adapters on the 7 main linear modules. Total trainable params: ~5M (vs 4B base).


In [ ]:
print(f"Attaching LoRA (r={LORA_R}, alpha={LORA_ALPHA}) ...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGETS,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",   # best VRAM for long ctx
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)


## 7. Load + filter + format dataset

~3-5 min cold (downloads ~12 GB), <30s warm. `num_proc=8` parallelizes across CPU cores.


In [ ]:
print(f"Loading {DATASET_NAME} ({DATASET_CONFIG}) ...")
ds = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
print(f"Raw rows: {len(ds):,}")

print("Formatting + filtering ...")
ds = ds.map(build_formatter(tokenizer), num_proc=8, desc="apply_chat_template")
ds = ds.filter(lambda r: r["text"] is not None, num_proc=8)
print(f"After filter: {len(ds):,}")

target = 200 if SMOKE_TEST else SUBSAMPLE_SIZE
if len(ds) > target:
    ds = ds.shuffle(seed=SEED).select(range(target))
print(f"After subsample: {len(ds):,}  (SMOKE_TEST={SMOKE_TEST})")

train_ds = ds


## 8. Sanity check — DON'T SKIP

Verify the chat template wrapped the data correctly. The three flags at the bottom MUST all be `True` before you proceed to training. The most common failure mode is the Qwen3 chat template stripping `<think>...</think>` from the assistant turn — if you see that, training will produce a model that no longer thinks.


In [ ]:
print("=" * 72)
print("First example (first 1500 chars):")
print("=" * 72)
print(train_ds[0]["text"][:1500])
print("=" * 72)

has_think_open  = "<think>"  in train_ds[0]["text"]
has_think_close = "</think>" in train_ds[0]["text"]
has_boxed       = "\\boxed{" in train_ds[0]["text"]
print(f"'<think>' in text:  {has_think_open}")
print(f"'</think>' in text: {has_think_close}")
print(f"'\\boxed{{' in text: {has_boxed}")

if not (has_think_open and has_think_close and has_boxed):
    print("\n⚠️  WARNING: expected markers missing — chat template may be stripping <think>.")
    print("    DO NOT proceed to training. Investigate and patch the formatter first.")
else:
    print("\n✅ All expected markers present. Safe to proceed.")


## 9. Trainer setup

`SFTConfig` encodes the entire recipe. `train_on_responses_only` masks the loss so only assistant tokens (including `<think>...</think>` and the final `\boxed{}`) contribute gradient — the user problem text and chat boilerplate get ignored.


In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=MAX_GRAD_NORM,
    weight_decay=WEIGHT_DECAY,
    optim="adamw_8bit",              # Unsloth's 8-bit AdamW
    bf16=True,
    fp16=False,
    seed=SEED,
    logging_steps=10,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    report_to="none",                # set "tensorboard" or "wandb" if you want curves
    max_length=MAX_SEQ_LEN,
    packing=False,                   # explicit; per-sample loss
    dataset_text_field="text",
    dataset_num_proc=4,
    push_to_hub=False,               # we do the upload ourselves at the end
    max_steps=10 if SMOKE_TEST else -1,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=sft_config,
)

# Mask loss to assistant tokens only.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


## 10. Step-count summary

Verify the math matches the canonical open-r1 reference (~3,672 optim steps) before committing to the long run.


In [ ]:
total_examples  = len(train_ds) * NUM_EPOCHS
total_opt_steps = total_examples // (PER_DEVICE_BS * GRAD_ACCUM)
warmup_steps    = int(total_opt_steps * WARMUP_RATIO)

print(f"Dataset size:      {len(train_ds):,}")
print(f"Epochs:            {NUM_EPOCHS}")
print(f"Effective batch:   {PER_DEVICE_BS * GRAD_ACCUM}")
print(f"Total fwd-back:    {total_examples:,}")
print(f"Total optim steps: {total_opt_steps:,}")
print(f"Warmup steps (3%): {warmup_steps:,}")
print(f"Save every:        {SAVE_STEPS} steps  (resume={RESUME})")
print(f"Output dir:        {OUTPUT_DIR}")
print(f"(Canonical open-r1 SFT runs ~3,672 optim steps for reference.)")


## 11. Train (the long one)

On A5000 24 GB: ~18h for the full run, ~10 min for the smoke test. JupyterLab keeps the cell running even if you close the tab, but a kernel restart (or pod preemption) kills it — checkpointing every 200 steps means you lose at most ~30 min and can resume by setting `RESUME = True` in cell 3 and re-running from cell 9 onward.


In [ ]:
print("Starting training ...")
trainer.train(resume_from_checkpoint=RESUME)
print("Training complete.")


## 12. Save final adapter

Writes the ~80 MB LoRA adapter to `<OUTPUT_DIR>/final/`. The 4-bit base weights are NOT saved (that's the point of LoRA — at inference you load the base + apply this adapter on top).


In [ ]:
final_dir = str(Path(OUTPUT_DIR) / "final")
print(f"Writing final adapter to {final_dir}")
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print("Saved.")


## 13. (Optional) Push adapter to HF Hub

Required for Track A5 (vLLM inference loads the adapter from the Hub). Set `PUSH_TO = '<user>/<repo>'` in cell 3, or run `hf upload` from a shell after training.


In [ ]:
if PUSH_TO:
    print(f"Uploading to https://huggingface.co/{PUSH_TO} ...")
    model.push_to_hub(PUSH_TO, private=False)
    tokenizer.push_to_hub(PUSH_TO, private=False)
    print("Push complete.")
else:
    print("PUSH_TO is None — skipping upload.")
    print(f"To push later from a shell: hf upload <user>/qwen3-4b-math-lora {Path(OUTPUT_DIR) / 'final'}")
